[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Modeling Without Joins


## What you will be able to do

Decide whether a thing belongs inside its parent document or in a collection of its own, and give a
reason rather than a preference. Say what the limit on embedding actually is, having hit it from
both directions. Join two collections with `$lookup`, index the field it joins on, and measure the
difference. Recognize the duplicated field that was copied for speed and is now wrong. And say what
is atomic here, which is one document, and what to do when you need two.


## The idea

### The problem

Relationally, this question does not exist: an order has lines, the lines are a table, and that is
the end of it. Here you choose, every time, and the choice is not reversible without a migration.

Embed and the read is one document with no join. Reference and the read is two queries or a
`$lookup`. Both are right, for different data, and the boundary between them is not a matter of
taste: it is sixteen megabytes.

### What the limit is

A BSON document may be at most 16MB. Not a row, not a field: the whole document, including every
embedded array. An array that grows without bound will reach it, and the day it does the write
stops, with the data already in the document intact and unreadable-to-grow.

### Why unbounded arrays are the rule

"Embed until the document will not fit" is the usual advice and it is not quite right, because a
document that will not fit is a document you cannot write at all. The useful form is: embed when
the array is **bounded** by something in the domain. An address has one country. An order has a
handful of lines. A post has an unbounded number of comments.

### Where this shows up

The comment thread inside the post. The event log inside the user. The "append one more" that has
worked for two years and stops working for the busiest document first, which is always the one that
matters.

### What this notebook covers

Embedding and referencing, side by side. The 16MB wall, hit in Python and on the server, which give
different errors. `$lookup`, indexed and not. The duplicated field that drifts. Then atomicity: one
document is free, two need a transaction.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

comment = {"by": "someone", "body": "x" * 900}

shop.posts.drop()
post = {"_id": 1, "title": "a post", "comments": [dict(comment) for _ in range(20000)]}
try:
    shop.posts.insert_one(post)
except pymongo.errors.DocumentTooLarge as error:
    print("embedded:  ", str(error).split(" - ")[0])

shop.comments.drop()
shop.comments.insert_many([{**comment, "post_id": 1} for _ in range(20000)])
print("referenced:", shop.comments.count_documents({"post_id": 1}), "comments")
client.close()
```

```
embedded:   BSON document too large (18769155 bytes)
referenced: 20000 comments
```

The same twenty thousand comments, twice. As an array inside the post they are a document too big to
write. As their own documents they are twenty thousand ordinary rows, and there is no number at
which that stops working.


## Setup

Nine imports, MongoDB, the boot cell, and three helpers.

- `pymongo` is the driver, `bson` measures a document, and `ReturnDocument` is for the last section
- `subprocess` and `os` install and start the server, `sys` names this Python, `time` waits
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

`size_of` encodes a document to BSON and gives its length, which is the number the 16MB limit
applies to. `work_done` runs `explain` over a pipeline, which is how the `$lookup` section is
measured. `COMMENT` is one comment of about a kilobyte, used throughout.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import bson
import pymongo
from pymongo import ReturnDocument

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

COMMENT = {"by": "someone", "body": "x" * 900}                      # about a kilobyte each


def failed(error):
    """A failure's real message, without the parts that change every run. The server reports a
    size in hex as well as in decimal, and a hex number is not worth committing to a notebook."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    message = message.split(" :: caused by :: ")[-1]
    while "(0x" in message:
        start = message.index("(0x")
        message = message[:start] + message[message.index(")", start) + 1:]
    return f"{type(error).__name__}: {message}"


def size_of(document):
    """How many bytes this document takes as BSON, which is what the limit counts."""
    return len(bson.encode(document))


def work_done(pipeline, collection="products"):
    """What the server read to run a pipeline, which is how a $lookup is judged."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        explained = shop.command("explain",
                                 {"aggregate": collection, "pipeline": pipeline, "cursor": {}},
                                 verbosity="executionStats")
        stats = explained.get("executionStats")
        if stats is None:
            stats = explained["stages"][0]["$cursor"]["executionStats"]
        return {"documents": stats["documentsExamined"] if "documentsExamined" in stats
                else stats["totalDocsExamined"],
                "index keys": stats["totalKeysExamined"]}


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products and", end=" ")
with pymongo.MongoClient(URI, tz_aware=True) as _client:
    print(_client.get_default_database().reviews.count_documents({}), "reviews")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products and 767 reviews
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### The same data, embedded and referenced


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

shop.orders_embedded.drop()
shop.orders_referenced.drop()
shop.order_lines.drop()

shop.orders_embedded.insert_one({
    "_id": 1, "reference": "AB-1",
    "lines": [{"sku": "LAP-000000", "qty": 1}, {"sku": "MOU-000003", "qty": 2}],
})

shop.orders_referenced.insert_one({"_id": 1, "reference": "AB-1"})
shop.order_lines.insert_many([{"order_id": 1, "sku": "LAP-000000", "qty": 1},
                              {"order_id": 1, "sku": "MOU-000003", "qty": 2}])

print("embedded, one query: ", shop.orders_embedded.find_one({"_id": 1}))
print()
print("referenced, two queries:")
print("  ", shop.orders_referenced.find_one({"_id": 1}))
print("  ", list(shop.order_lines.find({"order_id": 1}, {"_id": 0})))


embedded, one query:  {'_id': 1, 'reference': 'AB-1', 'lines': [{'sku': 'LAP-000000', 'qty': 1}, {'sku': 'MOU-000003', 'qty': 2}]}

referenced, two queries:
   {'_id': 1, 'reference': 'AB-1'}
   [{'order_id': 1, 'sku': 'LAP-000000', 'qty': 1}, {'order_id': 1, 'sku': 'MOU-000003', 'qty': 2}]


An order has a handful of lines and nobody edits a line without the order. That is the case for
embedding: one read, one write, and the lines cannot get out of step with the order because they are
the order.

### What embedding costs when the array is not bounded


In [3]:
for count in (1_000, 10_000, 17_000):
    post = {"_id": 1, "comments": [dict(COMMENT) for _ in range(count)]}
    print(f"  {count:6} comments: {size_of(post):>10,} bytes")

print()
print("the limit is 16777216 bytes, and a kilobyte each gets there in about sixteen thousand")


    1000 comments:    936,919 bytes
   10000 comments:  9,378,919 bytes
   17000 comments: 15,951,919 bytes

the limit is 16777216 bytes, and a kilobyte each gets there in about sixteen thousand


That is the arithmetic to do before choosing to embed. A thousand comments is a sixteenth of the
budget and fine. Seventeen thousand is over it, and the document cannot be written at all.

It is worse than a hard stop, because the whole document is read and written every time. A post with
ten thousand comments costs ten megabytes to read even when you only wanted the title.

### The wall, from both directions

Built in Python and handed to `insert_one`, the driver refuses before sending anything:


In [4]:
shop.posts.drop()
too_big = {"_id": 1, "comments": [dict(COMMENT) for _ in range(20_000)]}

try:
    shop.posts.insert_one(too_big)
except pymongo.errors.DocumentTooLarge as error:
    print(f"{type(error).__module__}.{type(error).__name__}:")
    print("  ", error)


pymongo.errors.DocumentTooLarge:
   BSON document too large (18769137 bytes) - the connected server supports BSON document sizes up to 16793598 bytes.


Grown on the server by `$push`, the document is under the limit until the update that takes it over,
and the failure comes from the server with a different message:


In [5]:
shop.posts.drop()
shop.posts.insert_one({"_id": 1, "comments": []})

pushed = 0
try:
    for _ in range(60):
        shop.posts.update_one({"_id": 1},
                              {"$push": {"comments": {"$each": [dict(COMMENT)] * 500}}})
        pushed += 500
except pymongo.errors.WriteError as error:
    print("stopped after", pushed, "comments")
    print("  ", failed(error).split(" First element")[0])

print("what is stored:", len(shop.posts.find_one({"_id": 1})["comments"]), "comments")


stopped after 17500 comments
   WriteError: BSONObj size: 16890919  is invalid. Size must be between 0 and 16793600(16MB)
what is stored: 17500 comments


Two things worth noticing. The stored document is fine and readable, at just under the limit, so
nothing is corrupt. And the failure is not a wall you hit once: every subsequent `$push` fails too,
so the feature simply stops working for that document while working for every other one.

That is what makes it a production incident rather than a bug report. It happens to the most active
document first.

### Referencing, and $lookup

The seeded `reviews` collection references `products` by `product_id`:


In [6]:
print("a product:", shop.products.find_one({"_id": 3}, {"name": 1}))
print("its reviews:", list(shop.reviews.find({"product_id": 3}, {"_id": 0, "product_id": 0})))


a product: {'_id': 3, 'name': 'Dalgo mouse 3'}
its reviews: [{'stars': 4, 'body': 'A review of Dalgo mouse 3'}, {'stars': 3, 'body': 'A review of Dalgo mouse 3'}]


Two queries, which is often the right answer: ask for the product, and ask for its reviews only if
you are going to show them.

`$lookup` does it in one, on the server:


In [7]:
joined = list(shop.products.aggregate([
    {"$match": {"_id": 3}},
    {"$lookup": {"from": "reviews", "localField": "_id",
                 "foreignField": "product_id", "as": "reviews"}},
    {"$project": {"_id": 0, "name": 1, "reviews.stars": 1}},
]))
print(joined)


[{'name': 'Dalgo mouse 3', 'reviews': [{'stars': 4}, {'stars': 3}]}]


`localField` is the field on the documents coming through the pipeline, `foreignField` is the field
on the other collection, and `as` is where the matches are put, always as an array.

### What $lookup costs without an index


In [8]:
pipeline = [{"$match": {"kind": "laptop"}},
            {"$lookup": {"from": "reviews", "localField": "_id",
                         "foreignField": "product_id", "as": "reviews"}}]

shop.reviews.drop_indexes()
print("no index on reviews.product_id:", work_done(pipeline))

shop.reviews.create_index("product_id", name="product_id_1")
print("with one:                      ", work_done(pipeline))


no index on reviews.product_id: {'documents': 867, 'index keys': 100}
with one:                       {'documents': 249, 'index keys': 249}


Without the index the server read every review in the collection to build a lookup table, on top of
the hundred laptops. With it, it read only the reviews that matched.

The numbers here are small because the collection is. The shape is what matters: without an index
the cost of a `$lookup` includes the whole foreign collection, every time the pipeline runs.

**Index the foreign field.** It is one line and it is the difference between a join that scales and
one that does not.

### The field you copied, and the day it is wrong

Duplicating a field into another document is a real technique, and it has a real cost:


In [9]:
shop.reviews.delete_many({"product_id": 3, "copied": True})
product = shop.products.find_one({"_id": 3})
shop.reviews.insert_one({"product_id": 3, "stars": 5, "copied": True,
                         "product_name": product["name"]})                # copied for speed

print("the review, with the name on it:",
      shop.reviews.find_one({"copied": True}, {"_id": 0, "product_name": 1}))

shop.products.update_one({"_id": 3}, {"$set": {"name": "Dalgo mouse 3, renamed"}})

print("after renaming the product:")
print("  the product says:", shop.products.find_one({"_id": 3}, {"_id": 0, "name": 1}))
print("  the review says: ", shop.reviews.find_one({"copied": True}, {"_id": 0, "product_name": 1}))


the review, with the name on it: {'product_name': 'Dalgo mouse 3'}
after renaming the product:
  the product says: {'name': 'Dalgo mouse 3, renamed'}
  the review says:  {'product_name': 'Dalgo mouse 3'}


Nothing failed. There are now two answers to "what is this product called", and any report built on
the reviews collection will use the stale one forever.

Copying a field is right when the copy is a **fact about the moment**, such as the price at which
something was actually sold, which should not change when the catalog does. It is wrong when it is
meant to be the same value as somewhere else, because keeping two copies in step is a job somebody
has to do on every write.


In [10]:
shop.products.update_one({"_id": 3}, {"$set": {"name": product["name"]}})   # put it back
shop.reviews.delete_many({"copied": True})
print("tidied:", shop.products.find_one({"_id": 3}, {"_id": 0, "name": 1}))


tidied: {'name': 'Dalgo mouse 3'}


### When to reach for which

| The relationship | Where it goes | Why |
|---|---|---|
| one, and always read together | embedded | one read, one write, never out of step |
| a bounded few, edited with the parent | embedded | an order's lines, an address's parts |
| unbounded, growing forever | its own collection | 16MB arrives, and arrives first for the busiest |
| large, and rarely read | its own collection | the parent is read whole, every time |
| shared between parents | its own collection | one copy, one place to change it |
| a value as it was at the time | copied, deliberately | the price sold at is not the price now |
| a value that must match another | referenced, never copied | two copies drift, silently |

The default is to embed. Reach for a separate collection when the array has no natural bound, when
the children are big, or when something else refers to them too, and index the field you will join
on before you need to.

### An order that keeps its own history, finished


In [11]:
def place(shop, reference, lines):
    """Lines are embedded, because an order has a handful and they are never read alone.
    The price is copied on purpose: it is what was charged, not what the catalog says now."""
    priced = []
    for line in lines:
        product = shop.products.find_one({"sku": line["sku"]}, {"price": 1, "name": 1})
        priced.append({**line, "name": product["name"], "price_then": product["price"]})

    order = {"reference": reference, "lines": priced,
             "total": round(sum(line["qty"] * line["price_then"] for line in priced), 2)}
    shop.orders.replace_one({"reference": reference}, order, upsert=True)
    return order["total"]


def add_event(shop, reference, what):
    """Events are referenced, because there is no limit to how many an order can collect."""
    shop.order_events.insert_one({"reference": reference, "what": what})
    return shop.order_events.count_documents({"reference": reference})


shop.orders.drop()
shop.order_events.drop()
shop.order_events.create_index("reference", name="reference_1")     # the field we will join on

total = place(shop, "AB-2", [{"sku": "LAP-000000", "qty": 1}, {"sku": "CAB-000004", "qty": 3}])
print("total charged:", total)

for what in ("placed", "paid", "packed", "shipped"):
    add_event(shop, "AB-2", what)

full = list(shop.orders.aggregate([
    {"$match": {"reference": "AB-2"}},
    {"$lookup": {"from": "order_events", "localField": "reference",
                 "foreignField": "reference", "as": "events"}},
    {"$project": {"_id": 0, "reference": 1, "total": 1, "lines": {"$size": "$lines"},
                  "events": "$events.what"}},
]))
print("the whole order:", full)


total charged: 5360.78
the whole order: [{'reference': 'AB-2', 'total': 5360.78, 'lines': 2, 'events': ['placed', 'paid', 'packed', 'shipped']}]


Two lines embedded and four events referenced, in one order, for reasons that are different and
both stated in the code. The lines are bounded and belong to the order. The events are not bounded:
an order that goes wrong can collect hundreds, and nothing should stop working when it does.

`price_then` is copied deliberately and named so that nobody mistakes it for the current price.

### Where each part came from

| In the order | What it relies on | The section that showed it |
|---|---|---|
| `lines` embedded | a bounded array read with its parent | The same data, embedded and referenced |
| `order_events` referenced | an array with no natural bound | What embedding costs |
| the index on `reference` | `$lookup` reading the whole collection without one | What $lookup costs |
| `price_then` | a copy that is a fact about the moment | The field you copied |
| `$size` and `$events.what` | `$project` computing from a joined array | **The Aggregation Pipeline** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/10-modeling-without-joins-solutions.ipynb).

**1.** Measure how many kilobyte comments fit in one document.


In [12]:
# your code here


**2.** Try to insert a document over the limit and print the error.


In [13]:
# your code here


**3.** Join a product to its reviews with `$lookup`.


In [14]:
# your code here


**4.** Measure a `$lookup` with and without an index on the foreign field.


In [15]:
# your code here


**5.** Store the same lines embedded and referenced, and read both back.


In [16]:
# your code here


**6.** Change a value that was copied into another collection and show the two disagree.


In [17]:
# your code here


## Common errors

### pymongo.errors.DocumentTooLarge: BSON document too large


In [18]:
shop.posts.drop()
shop.posts.insert_one({"_id": 1, "comments": [dict(COMMENT) for _ in range(20_000)]})


DocumentTooLarge: BSON document too large (18769137 bytes) - the connected server supports BSON document sizes up to 16793598 bytes.

PyMongo raised this, not the server: it encoded the document, saw the size, and refused to send it.
That is why the number in the message is the document's size rather than anything about the
collection.

Nothing was written. The fix is not a bigger document, because there is no bigger document; it is a
different shape:


In [19]:
shop.posts.drop()
shop.comments.drop()
shop.comments.create_index("post_id", name="post_id_1")

shop.posts.insert_one({"_id": 1, "title": "a post"})
shop.comments.insert_many([{**COMMENT, "post_id": 1} for _ in range(20_000)])

print("the post:    ", shop.posts.find_one({"_id": 1}))
print("its comments:", shop.comments.count_documents({"post_id": 1}))
print("and reading the post no longer reads twenty megabytes")


the post:     {'_id': 1, 'title': 'a post'}
its comments: 20000
and reading the post no longer reads twenty megabytes


### pymongo.errors.WriteError: BSONObj size is invalid


In [20]:
shop.grow.drop()
shop.grow.insert_one({"_id": 1, "comments": [dict(COMMENT) for _ in range(16_000)]})

try:
    shop.grow.update_one({"_id": 1},
                         {"$push": {"comments": {"$each": [dict(COMMENT)] * 2000}}})
except pymongo.errors.WriteError as error:
    print(failed(error).split(" First element")[0])


WriteError: BSONObj size: 16890919  is invalid. Size must be between 0 and 16793600(16MB)


The same limit, reported by the server rather than the driver, because this time the document PyMongo
sent was small: it was the `$push` that would have made the result too big.

The number in the message is the size the document **would have been**, and the document on disk is
unchanged. Note how different the wording is from the driver's version, which matters when you are
searching for it in a log.

### No error: the copy that drifted


In [21]:
shop.catalog.drop()
shop.cart.drop()
shop.catalog.insert_one({"_id": "LAP-1", "name": "a laptop", "price": 1000})
shop.cart.insert_one({"_id": 1, "sku": "LAP-1", "name": "a laptop", "price": 1000})

shop.catalog.update_one({"_id": "LAP-1"}, {"$set": {"price": 1200, "name": "a better laptop"}})

print("catalog:", shop.catalog.find_one({"_id": "LAP-1"}, {"_id": 0}))
print("cart:   ", shop.cart.find_one({"_id": 1}, {"_id": 0, "sku": 0}))
print()
print("one of these is right and nothing knows which")


catalog: {'name': 'a better laptop', 'price': 1200}
cart:    {'name': 'a laptop', 'price': 1000}

one of these is right and nothing knows which


Both fields were copied for the same reason and only one of them should have been. The **price** in
the cart is correct as it stands: it is what the customer was quoted, and changing it because the
catalog changed would be wrong. The **name** is simply stale.

Name the ones you mean to freeze, and look the others up:


In [22]:
shop.cart.drop()
shop.cart.insert_one({"_id": 1, "sku": "LAP-1", "price_quoted": 1000})   # frozen, and says so

line = shop.cart.find_one({"_id": 1})
product = shop.catalog.find_one({"_id": line["sku"]}, {"_id": 0, "name": 1, "price": 1})
print("quoted at:", line["price_quoted"], "| the catalog now says:", product)
print("both true, and the field names say which is which")


quoted at: 1000 | the catalog now says: {'name': 'a better laptop', 'price': 1200}
both true, and the field names say which is which


### No error: two documents changed one at a time


In [23]:
shop.accounts.drop()
shop.accounts.insert_many([{"_id": "a", "n": 100}, {"_id": "b", "n": 0}])

shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}})
print("after the first write:", list(shop.accounts.find().sort("_id")))
print("if the process died here, ten units would have vanished")
shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}})
print("after the second:     ", list(shop.accounts.find().sort("_id")))


after the first write: [{'_id': 'a', 'n': 90}, {'_id': 'b', 'n': 0}]
if the process died here, ten units would have vanished
after the second:      [{'_id': 'a', 'n': 90}, {'_id': 'b', 'n': 10}]


Every single-document write here is atomic, and that guarantee stops exactly at the document
boundary. Two writes are two operations with a gap, and the gap is where a crash leaves the data
disagreeing with itself.

This is the strongest argument for embedding that exists: things that must change together and live
in one document change together for free. When they cannot, **Bulk Writes and Transactions** is the
tool, and it costs a session and a replica set:


In [24]:
shop.accounts.drop()
shop.accounts.insert_many([{"_id": "a", "n": 100}, {"_id": "b", "n": 0}])


def move(session):
    shop.accounts.update_one({"_id": "a"}, {"$inc": {"n": -10}}, session=session)
    shop.accounts.update_one({"_id": "b"}, {"$inc": {"n": 10}}, session=session)


with client.start_session() as session:
    session.with_transaction(move)

print("both, or neither:", list(shop.accounts.find().sort("_id")))


both, or neither: [{'_id': 'a', 'n': 90}, {'_id': 'b', 'n': 10}]


In [25]:
for name in ("posts", "comments", "grow", "catalog", "cart", "accounts",
             "orders", "order_events", "orders_embedded", "orders_referenced", "order_lines"):
    shop[name].drop()
client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- A BSON document may be at most 16MB, including everything embedded in it. That is the real
  boundary between embedding and referencing.
- Embed what is bounded by the domain and read with its parent. Reference what grows without
  limit, what is large, or what something else also refers to.
- An oversize `insert_one` raises `pymongo.errors.DocumentTooLarge` from the driver, before
  anything is sent. The same document grown by `$push` fails on the server with a
  `BSONObj size ... is invalid` `WriteError`, and the stored document is left intact and unable
  to grow.
- A post with ten thousand comments costs ten megabytes to read even when you wanted the title.
- `$lookup` joins on `localField` and `foreignField` and puts matches in `as`, always as an array.
  Without an index on the foreign field the whole foreign collection is read.
- A field copied into another document is right when it records what was true at a moment, and
  wrong when it is meant to equal something elsewhere. Name it so the difference is visible.
- One document's write is atomic. Two documents need a transaction, which is why embedding things
  that must agree is the cheapest correctness there is.


## What is next

**Beanie Documents** starts the second half of the guide: a Pydantic model that is also a
collection, `init_beanie`, and the first `await` in the guide, along with the event loop errors
that come with it.


---

&#8592; **Previous:** [The Aggregation Pipeline](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/09-the-aggregation-pipeline.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Beanie Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/11-beanie-documents.ipynb) &#8594;
